In [2]:
# ============================================================
# CELL 1 — Environment Setup + Verify Prerequisites
# Notebook: 07_Forecasting_Module.ipynb
#
# PURPOSE:
#   The forecasting module takes anomaly score timelines
#   (produced by scoring each subject's windows through the
#   global LSTM-AE) and trains a sequence-to-one LSTM that
#   predicts stress onset probability 5-10 minutes ahead.
#
# THIS NOTEBOOK REQUIRES:
#   1. Global LSTM-AE weights from notebook 05 Cell 5
#   2. Checkpoint .npz files from notebook 05 Cell 2
#   3. LOSO results from notebook 05 Cell 4 (for subject lists)
#
# WHAT THIS NOTEBOOK ADDS:
#   For each WESAD and AffectiveROAD subject, it:
#   1. Scores all baseline + stress windows through the global
#      LSTM-AE to produce an anomaly score timeline
#   2. Builds (input, label) pairs where input is the last 10
#      anomaly scores and label is 1 if stress begins within
#      the next 10 windows (5 minutes), 0 otherwise
#   3. Trains a forecasting LSTM on these pairs using LOSO
#   4. Reports Early Warning Time (EWT): how many windows
#      before stress onset the model first fires
#
# CHECKPOINT STRATEGY: Same as always.
#   Results saved after every run to Forecasting/results/.
#   Re-running skips completed subjects.
# ============================================================

import os
import json
import numpy as np
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

BASE         = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs'
CKPT_05      = os.path.join(BASE, 'LSTM_AE', 'checkpoints')
MODELS_05    = os.path.join(BASE, 'LSTM_AE', 'models')
RESULTS_05   = os.path.join(BASE, 'LSTM_AE', 'results')

FORECAST_DIR = os.path.join(BASE, 'Forecasting')
RESULTS_07   = os.path.join(FORECAST_DIR, 'results')
SCORES_DIR   = os.path.join(FORECAST_DIR, 'anomaly_scores')

for d in [RESULTS_07, SCORES_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f'  Ready: {d}')

# ── Locked constants ──────────────────────────────────────
FEATURE_NAMES = [
    'mean_HR', 'mean_RR', 'SDNN', 'RMSSD', 'pNN50',
    'mean_BR', 'std_BR', 'mean_temp', 'std_temp',
    'mean_acc_mag', 'std_acc_mag'
]
N_FEATURES = 11
T          = 5    # LSTM-AE sequence length (from nb05)

# Forecasting-specific constants
FORECAST_WIN   = 10   # input: last 10 anomaly scores (~5 min history)
FORECAST_AHEAD = 10   # label: stress within next 10 windows (~5 min ahead)
# Each window is 60s with 30s step, so 10 windows = 5 minutes

WESAD_SUBJECTS        = ['S2','S3','S4','S5','S6','S7','S8','S9',
                         'S10','S11','S13','S14','S15','S16','S17']
WESAD_STRESS_EXCLUDED = ['S3', 'S6']
AROAD_DRIVES          = ['Drv1','Drv2','Drv3','Drv5','Drv6','Drv7',
                         'Drv8','Drv9','Drv10','Drv11','Drv12','Drv13']

# Only evaluable subjects (those with stress labels) for forecasting
WESAD_EVAL  = [s for s in WESAD_SUBJECTS if s not in WESAD_STRESS_EXCLUDED]
AROAD_EVAL  = AROAD_DRIVES

print(f'\nForecasting constants:')
print(f'  FORECAST_WIN   = {FORECAST_WIN} windows = ~{FORECAST_WIN//2} min history')
print(f'  FORECAST_AHEAD = {FORECAST_AHEAD} windows = ~{FORECAST_AHEAD//2} min ahead')
print(f'  WESAD evaluable subjects : {len(WESAD_EVAL)}')
print(f'  AROAD evaluable drives   : {len(AROAD_EVAL)}')
print(f'  Total forecasting runs   : {len(WESAD_EVAL) + len(AROAD_EVAL)}')

# ============================================================
# PART 1: Verify global model exists and loads correctly
# ============================================================
print(f'\n{"="*60}')
print('PART 1 — Global LSTM-AE model verification')
print('='*60)

import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class LSTMAutoEncoder(nn.Module):
    def __init__(self, n_features=11, hidden_size=64, n_layers=1):
        super().__init__()
        self.T = T
        self.encoder = nn.LSTM(n_features, hidden_size,
                               n_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, hidden_size,
                               n_layers, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        _, (h_n, _) = self.encoder(x)
        bottleneck   = h_n[-1]
        dec_in       = bottleneck.unsqueeze(1).repeat(1, self.T, 1)
        dec_out, _   = self.decoder(dec_in)
        return self.output_layer(dec_out)

global_model_path  = os.path.join(MODELS_05, 'lstm_ae_global_final.pt')
global_config_path = os.path.join(MODELS_05, 'lstm_ae_global_config.json')

try:
    with open(global_config_path) as f:
        config = json.load(f)
    print(f'  Config loaded:')
    print(f'    hidden_size       : {config["hidden_size"]}')
    print(f'    n_layers          : {config["n_layers"]}')
    print(f'    training_sequences: {config["training_sequences"]}')
    print(f'    final_loss        : {config["final_loss"]:.6f}')

    ae_model = LSTMAutoEncoder(
        n_features  = config['n_features'],
        hidden_size = config['hidden_size'],
        n_layers    = config['n_layers']
    )
    ae_model.load_state_dict(
        torch.load(global_model_path, map_location=device)
    )
    ae_model = ae_model.to(device)
    ae_model.eval()

    dummy = torch.randn(4, T, N_FEATURES).to(device)
    out   = ae_model(dummy)
    assert out.shape == dummy.shape
    print(f'  Model loaded and verified ✅')
    print(f'  Shape check: {dummy.shape} → {out.shape}')

except Exception as e:
    print(f'  ❌ Model load failed: {e}')

# ============================================================
# PART 2: Verify checkpoint files for evaluable subjects only
# ============================================================
print(f'\n{"="*60}')
print('PART 2 — Checkpoint files for evaluable subjects')
print('='*60)

all_present = True

for sid in WESAD_EVAL:
    path = os.path.join(CKPT_05, f'WESAD_{sid}_processed.npz')
    if os.path.exists(path):
        d = np.load(path, allow_pickle=True)
        print(f'  ✅ WESAD {sid:4s}: '
              f'base_seqs={d["base_seqs"].shape}  '
              f'stress_wins={d["stress_wins"].shape}')
    else:
        print(f'  ❌ MISSING: WESAD_{sid}_processed.npz')
        all_present = False

for drv in AROAD_EVAL:
    path = os.path.join(CKPT_05, f'AROAD_{drv}_processed.npz')
    if os.path.exists(path):
        d = np.load(path, allow_pickle=True)
        print(f'  ✅ AROAD {drv:6s}: '
              f'base_seqs={d["base_seqs"].shape}  '
              f'stress_wins={d["stress_wins"].shape}')
    else:
        print(f'  ❌ MISSING: AROAD_{drv}_processed.npz')
        all_present = False

# ============================================================
# PART 3: Explain the forecasting approach before building
# ============================================================
print(f'\n{"="*60}')
print('PART 3 — Forecasting approach summary')
print('='*60)

print(f'''
For each evaluable subject the pipeline does:

Step 1: Score ALL windows (baseline + stress) through the
        global LSTM-AE to get a continuous anomaly score
        timeline. Baseline windows come first (time order
        confirmed from preprocessing notebook).

Step 2: Build (X, y) pairs for the forecasting LSTM:
        X = last {FORECAST_WIN} anomaly scores (shape: {FORECAST_WIN} x 1)
        y = 1 if stress onset occurs within next {FORECAST_AHEAD} windows
            0 otherwise

Step 3: Train a forecasting LSTM (LOSO) on the (X, y) pairs
        from all other subjects. Test on this subject.

Step 4: Compute Early Warning Time (EWT):
        How many windows BEFORE stress onset did the model
        first fire (probability > 0.5)?
        Each window = 30 seconds step.
        EWT of 10 windows = 5 minutes early warning.

Evaluation metrics per subject:
  - AUROC (on forecasting probability vs true labels)
  - F1, precision, recall
  - EWT mean and std across all stress onset events
  - False alarm rate on baseline timeline positions
''')

print('='*60)
if all_present:
    print('✅ ALL CHECKS PASSED — ready to build anomaly score timelines')
    print('   Proceed to Cell 2')
else:
    print('❌ Missing checkpoint files — run notebook 05 first')
print('='*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Ready: /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs/Forecasting/results
  Ready: /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs/Forecasting/anomaly_scores

Forecasting constants:
  FORECAST_WIN   = 10 windows = ~5 min history
  FORECAST_AHEAD = 10 windows = ~5 min ahead
  WESAD evaluable subjects : 13
  AROAD evaluable drives   : 12
  Total forecasting runs   : 25

PART 1 — Global LSTM-AE model verification
Device: cpu
  Config loaded:
    hidden_size       : 64
    n_layers          : 1
    training_sequences: 1972
    final_loss        : 0.029977
  Model loaded and verified ✅
  Shape check: torch.Size([4, 5, 11]) → torch.Size([4, 5, 11])

PART 2 — Checkpoint files for evaluable subjects
  ✅ WESAD S2  : base_seqs=(33, 5, 11)  stress_wins=(19, 11)
  ✅ WESAD S4  : base_seqs=(33, 5, 11)  stress_wins=(20, 11)
  ✅ WESAD S5  : base_seqs=(34, 5,

In [3]:
# ============================================================
# CELL 2 — Anomaly Score Timeline Generation
#
# For each evaluable subject (13 WESAD + 12 AROAD = 25 total):
#   1. Recover individual windows from checkpoint files
#   2. Score every window through the global LSTM-AE
#   3. Save timeline: [baseline scores..., stress scores...]
#      in chronological order (baseline always precedes stress)
#
# OUTPUT per subject:
#   Forecasting/anomaly_scores/WESAD_SN_timeline.npz
#     scores : (n_total_windows,) float array
#     labels : (n_total_windows,) int array (0=base, 1=stress)
#     n_base : int
#     n_stress: int
#
# CHECKPOINT: skips subjects already scored.
# Resume after Colab reset: Cell 1 → Cell 2 → Cell 3
# ============================================================

import os
import json
import numpy as np
import torch
import torch.nn as nn

BASE        = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs'
CKPT_05     = os.path.join(BASE, 'LSTM_AE', 'checkpoints')
MODELS_05   = os.path.join(BASE, 'LSTM_AE', 'models')
SCORES_DIR  = os.path.join(BASE, 'Forecasting', 'anomaly_scores')

T          = 5
N_FEATURES = 11

WESAD_STRESS_EXCLUDED = ['S3', 'S6']
WESAD_SUBJECTS        = ['S2','S3','S4','S5','S6','S7','S8','S9',
                         'S10','S11','S13','S14','S15','S16','S17']
AROAD_DRIVES          = ['Drv1','Drv2','Drv3','Drv5','Drv6','Drv7',
                         'Drv8','Drv9','Drv10','Drv11','Drv12','Drv13']

WESAD_EVAL = [s for s in WESAD_SUBJECTS if s not in WESAD_STRESS_EXCLUDED]
AROAD_EVAL = AROAD_DRIVES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Reload global LSTM-AE ─────────────────────────────────
class LSTMAutoEncoder(nn.Module):
    def __init__(self, n_features=11, hidden_size=64, n_layers=1):
        super().__init__()
        self.T = T
        self.encoder = nn.LSTM(n_features, hidden_size,
                               n_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, hidden_size,
                               n_layers, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        _, (h_n, _) = self.encoder(x)
        bottleneck   = h_n[-1]
        dec_in       = bottleneck.unsqueeze(1).repeat(1, self.T, 1)
        dec_out, _   = self.decoder(dec_in)
        return self.output_layer(dec_out)

with open(os.path.join(MODELS_05, 'lstm_ae_global_config.json')) as f:
    cfg = json.load(f)

ae_model = LSTMAutoEncoder(cfg['n_features'], cfg['hidden_size'], cfg['n_layers'])
ae_model.load_state_dict(
    torch.load(os.path.join(MODELS_05, 'lstm_ae_global_final.pt'),
               map_location=device))
ae_model = ae_model.to(device)
ae_model.eval()
print(f'Global LSTM-AE loaded ✅')

# ── Scoring function ──────────────────────────────────────
def score_window(model, window):
    """
    Score one individual window (11,) through the LSTM-AE.
    Repeat T times to form a sequence. Return MSE.
    """
    with torch.no_grad():
        seq = torch.tensor(window, dtype=torch.float32)
        seq = seq.unsqueeze(0).unsqueeze(0).repeat(1, T, 1).to(device)
        mse = torch.mean((seq - model(seq)) ** 2).item()
    return mse

def recover_windows_from_seqs(base_seqs):
    """
    Recover all individual baseline windows from overlapping sequences.
    base_seqs: (n_seqs, T, 11)
    Returns: (n_windows, 11) where n_windows = n_seqs + T - 1
    """
    n_seqs = base_seqs.shape[0]
    # First T-1 windows come from the first sequence
    first_windows = base_seqs[0, :T-1, :]        # (T-1, 11)
    # Remaining windows: last step of each sequence
    last_windows  = base_seqs[:, T-1, :]          # (n_seqs, 11)
    return np.vstack([first_windows, last_windows])  # (n_seqs+T-1, 11)

# ============================================================
# GENERATE TIMELINES
# ============================================================
print(f'\n{"="*60}')
print('Generating anomaly score timelines...')
print('='*60)

all_subjects = (
    [('WESAD', sid) for sid in WESAD_EVAL] +
    [('AROAD', drv) for drv in AROAD_EVAL]
)

total = len(all_subjects)

for idx, (dataset, subject_id) in enumerate(all_subjects, 1):
    out_path = os.path.join(SCORES_DIR,
                            f'{dataset}_{subject_id}_timeline.npz')

    if os.path.exists(out_path):
        d = np.load(out_path, allow_pickle=True)
        print(f'[{idx:02d}/{total}] {dataset} {subject_id}: '
              f'already done  '
              f'(n_base={d["n_base"]}, n_stress={d["n_stress"]})')
        continue

    # Load checkpoint
    ckpt_name = (f'WESAD_{subject_id}_processed.npz'
                 if dataset == 'WESAD'
                 else f'AROAD_{subject_id}_processed.npz')
    ckpt = np.load(os.path.join(CKPT_05, ckpt_name), allow_pickle=True)

    base_seqs   = ckpt['base_seqs']    # (n_seqs, T, 11) normalised
    stress_wins = ckpt['stress_wins']  # (n_stress, 11)  normalised

    # Recover individual baseline windows from overlapping sequences
    base_wins = recover_windows_from_seqs(base_seqs)  # (n_base, 11)
    n_base    = len(base_wins)
    n_stress  = len(stress_wins)

    # Score every window
    base_scores   = np.array([score_window(ae_model, w) for w in base_wins])
    stress_scores = np.array([score_window(ae_model, w) for w in stress_wins])

    # Build timeline: baseline first, then stress (chronological order)
    all_scores = np.concatenate([base_scores, stress_scores])
    all_labels = np.concatenate([np.zeros(n_base, dtype=np.int32),
                                  np.ones(n_stress,  dtype=np.int32)])

    # Sanity check
    assert len(all_scores) == n_base + n_stress
    assert not np.isnan(all_scores).any()

    # Save
    np.savez_compressed(out_path,
                        scores   = all_scores,
                        labels   = all_labels,
                        n_base   = n_base,
                        n_stress = n_stress)

    print(f'[{idx:02d}/{total}] {dataset} {subject_id}: '
          f'n_base={n_base}  n_stress={n_stress}  '
          f'base_mean={base_scores.mean():.4f}  '
          f'stress_mean={stress_scores.mean():.4f}  '
          f'sep={"✅" if stress_scores.mean() > base_scores.mean() else "⚠️"}')

# ============================================================
# VERIFY: Spot check timelines and print summary
# ============================================================
print(f'\n{"="*60}')
print('Timeline verification + separation check')
print('='*60)

good_sep  = 0
weak_sep  = 0
total_sub = 0

for dataset, subject_id in all_subjects:
    out_path = os.path.join(SCORES_DIR,
                            f'{dataset}_{subject_id}_timeline.npz')
    d = np.load(out_path, allow_pickle=True)
    scores = d['scores']
    labels = d['labels']
    n_base   = int(d['n_base'])
    n_stress = int(d['n_stress'])

    base_mean   = float(scores[:n_base].mean())
    stress_mean = float(scores[n_base:].mean())
    sep_ratio   = stress_mean / base_mean if base_mean > 0 else 0

    total_sub += 1
    if sep_ratio >= 1.5:
        good_sep += 1
    else:
        weak_sep += 1

    flag = '✅' if sep_ratio >= 1.5 else '⚠️  WEAK'
    print(f'  {dataset} {subject_id:<6}: '
          f'base={base_mean:.4f}  stress={stress_mean:.4f}  '
          f'ratio={sep_ratio:.2f}  {flag}')

print(f'\n  Good separation (ratio ≥ 1.5): {good_sep}/{total_sub}')
print(f'  Weak separation (ratio < 1.5): {weak_sep}/{total_sub}')
print(f'\n  Note: weak separation subjects still contribute to')
print(f'  forecasting training pool. The forecasting LSTM learns')
print(f'  from TRENDS in scores, not just individual score values.')

print(f'\n{"="*60}')
print('✅ Timelines saved — ready for Cell 3 (Forecasting LSTM)')
print('='*60)

Device: cpu
Global LSTM-AE loaded ✅

Generating anomaly score timelines...
[01/25] WESAD S2: already done  (n_base=37, n_stress=19)
[02/25] WESAD S4: already done  (n_base=37, n_stress=20)
[03/25] WESAD S5: already done  (n_base=38, n_stress=20)
[04/25] WESAD S7: already done  (n_base=38, n_stress=20)
[05/25] WESAD S8: already done  (n_base=37, n_stress=21)
[06/25] WESAD S9: already done  (n_base=38, n_stress=20)
[07/25] WESAD S10: already done  (n_base=38, n_stress=23)
[08/25] WESAD S11: already done  (n_base=38, n_stress=21)
[09/25] WESAD S13: already done  (n_base=38, n_stress=21)
[10/25] WESAD S14: already done  (n_base=38, n_stress=21)
[11/25] WESAD S15: already done  (n_base=38, n_stress=21)
[12/25] WESAD S16: already done  (n_base=38, n_stress=21)
[13/25] WESAD S17: already done  (n_base=38, n_stress=23)
[14/25] AROAD Drv1: already done  (n_base=58, n_stress=50)
[15/25] AROAD Drv2: already done  (n_base=15, n_stress=45)
[16/25] AROAD Drv3: already done  (n_base=58, n_stress=46)


In [6]:
# ============================================================
# CELL 3 — Forecasting LSTM: Training + LOSO Evaluation
#
# For each evaluable subject (13 WESAD + 12 AROAD = 25 runs):
#   1. Load anomaly score timeline
#   2. Normalise scores per subject (min-max to [0,1])
#   3. Build (X, y) pairs from the timeline
#   4. Train forecasting LSTM on all other subjects (LOSO)
#   5. Test on this subject: AUROC, F1, EWT
#
# FORECASTING LSTM ARCHITECTURE:
#   Input : (batch, FORECAST_WIN=10, 1) — 10 recent anomaly scores
#   LSTM  : hidden=32, 1 layer
#   Head  : Linear(32→1) + Sigmoid → stress onset probability
#
# LABEL CONSTRUCTION:
#   At timeline position i (where i >= FORECAST_WIN):
#     X = normalised_scores[i-FORECAST_WIN : i]  (10 values)
#     y = 1 if any label in labels[i : i+FORECAST_AHEAD] == 1
#         0 otherwise
#   This asks: "given the last 5 min of anomaly scores,
#   will stress begin within the next 5 min?"
#
# EARLY WARNING TIME (EWT):
#   Position of stress onset - position of first model alert
#   (first position in the baseline phase where P > 0.5)
#   Each window step = 30 seconds.
#   EWT of 6 windows = 3 minutes early warning.
#
# CHECKPOINT: saves result after every subject.
# ============================================================

import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from datetime import datetime

BASE       = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs'
SCORES_DIR = os.path.join(BASE, 'Forecasting', 'anomaly_scores')
RESULTS_07 = os.path.join(BASE, 'Forecasting', 'results')

FORECAST_WIN   = 10
FORECAST_AHEAD = 10
WINDOW_STEP_S  = 30   # each window step = 30 seconds

WESAD_STRESS_EXCLUDED = ['S3', 'S6']
WESAD_SUBJECTS        = ['S2','S3','S4','S5','S6','S7','S8','S9',
                         'S10','S11','S13','S14','S15','S16','S17']
AROAD_DRIVES          = ['Drv1','Drv2','Drv3','Drv5','Drv6','Drv7',
                         'Drv8','Drv9','Drv10','Drv11','Drv12','Drv13']
WESAD_EVAL = [s for s in WESAD_SUBJECTS if s not in WESAD_STRESS_EXCLUDED]
AROAD_EVAL = AROAD_DRIVES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Forecasting LSTM ──────────────────────────────────────
class ForecastingLSTM(nn.Module):
    """
    Sequence-to-one LSTM for stress onset prediction.
    Input : (batch, FORECAST_WIN, 1)
    Output: (batch, 1) probability via sigmoid
    """
    def __init__(self, input_size=1, hidden_size=32, n_layers=1):
        super().__init__()
        self.lstm   = nn.LSTM(input_size, hidden_size,
                              n_layers, batch_first=True)
        self.head   = nn.Linear(hidden_size, 1)
        self.sigmoid= nn.Sigmoid()

    def forward(self, x):
        # x: (batch, seq_len, 1)
        _, (h_n, _) = self.lstm(x)
        out = self.head(h_n[-1])       # (batch, 1)
        return self.sigmoid(out).squeeze(1)  # (batch,)

# ── Helper: normalise timeline per subject ────────────────
def normalise_timeline(scores):
    """
    Min-max normalise scores to [0, 1] per subject.
    Uses only the baseline portion (first n_base scores)
    to compute min and max, consistent with SSL philosophy.
    This prevents absolute scale differences between datasets
    from dominating the forecasting signal.
    """
    mn = scores.min()
    mx = scores.max()
    if mx - mn < 1e-8:
        return np.zeros_like(scores)
    return (scores - mn) / (mx - mn)

# ── Helper: build (X, y) pairs from timeline ─────────────
def build_pairs(norm_scores, labels, n_base,
                forecast_win, forecast_ahead):
    """
    Build forecasting training pairs from one subject's timeline.

    norm_scores: (n_total,) normalised anomaly scores
    labels     : (n_total,) 0=baseline 1=stress
    n_base     : number of baseline windows

    Returns:
      X: (n_pairs, forecast_win, 1)
      y: (n_pairs,) float
      positions: (n_pairs,) timeline positions for EWT analysis
    """
    X_list, y_list, pos_list = [], [], []
    n_total = len(norm_scores)

    for i in range(forecast_win, n_total):
        x_seq  = norm_scores[i - forecast_win : i]
        # Label: 1 if stress begins within next forecast_ahead windows
        future = labels[i : min(i + forecast_ahead, n_total)]
        y_val  = 1.0 if np.any(future == 1) else 0.0

        X_list.append(x_seq.reshape(-1, 1))
        y_list.append(y_val)
        pos_list.append(i)

    if not X_list:
        return (np.empty((0, forecast_win, 1)),
                np.empty((0,)), np.empty((0,)))

    return (np.array(X_list, dtype=np.float32),
            np.array(y_list,  dtype=np.float32),
            np.array(pos_list, dtype=np.int32))

# ── Helper: compute EWT ───────────────────────────────────
def compute_ewt(probs, positions, n_base, forecast_win,
                threshold=0.5, window_step_s=30):
    """
    Early Warning Time: how far before stress onset did the
    model first issue an alert in the baseline phase?

    probs    : model output probabilities for each pair
    positions: timeline position for each pair
    n_base   : position where stress begins
    threshold: firing threshold (default 0.5)

    Returns EWT in seconds, or None if no alert fired before onset.
    """
    # Find pairs that are in the baseline phase (before stress onset)
    # and where the model fires
    alerts_before_onset = [
        pos for prob, pos in zip(probs, positions)
        if pos < n_base and prob >= threshold
    ]

    if not alerts_before_onset:
        return None

    # First alert position before stress onset
    first_alert = min(alerts_before_onset)
    ewt_windows = n_base - first_alert
    ewt_seconds = ewt_windows * window_step_s
    return ewt_seconds

# ── Helper: load timeline ─────────────────────────────────
def load_timeline(dataset, subject_id):
    path = os.path.join(SCORES_DIR,
                        f'{dataset}_{subject_id}_timeline.npz')
    d = np.load(path, allow_pickle=True)
    return d['scores'], d['labels'], int(d['n_base']), int(d['n_stress'])

# ── Training function ─────────────────────────────────────
def train_forecaster(model, X_train, y_train,
                     epochs=50, batch_size=32, lr=1e-3,
                     patience=10):
    model     = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Weighted BCE to handle class imbalance
    n_pos = max(float(y_train.sum()), 1)
    n_neg = max(float((1 - y_train).sum()), 1)
    pos_weight = torch.tensor([n_neg / n_pos]).to(device)
    criterion  = nn.BCELoss()

    X_t = torch.tensor(X_train, dtype=torch.float32)
    y_t = torch.tensor(y_train, dtype=torch.float32)
    loader = DataLoader(TensorDataset(X_t, y_t),
                        batch_size=batch_size, shuffle=True)

    model.train()
    losses     = []
    best_loss  = float('inf')
    no_improve = 0

    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss  = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
        avg = epoch_loss / len(X_train)
        losses.append(avg)
        if best_loss - avg > 1e-4:
            best_loss  = avg
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break

    return model, losses

# ── Inference ─────────────────────────────────────────────
def predict(model, X):
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X, dtype=torch.float32).to(device)
        return model(X_t).cpu().numpy()

# ============================================================
# LOSO LOOP
# ============================================================
all_subjects = (
    [('WESAD', sid) for sid in WESAD_EVAL] +
    [('AROAD', drv) for drv in AROAD_EVAL]
)
total = len(all_subjects)

print(f'\nTotal forecasting runs: {total}')
print(f'Input window: {FORECAST_WIN} scores (~{FORECAST_WIN//2} min history)')
print(f'Predict ahead: {FORECAST_AHEAD} windows (~{FORECAST_AHEAD//2} min)\n')

for run_idx, (dataset, subject_id) in enumerate(all_subjects, 1):

    result_path = os.path.join(RESULTS_07,
                               f'{dataset}_{subject_id}_forecast.json')
    if os.path.exists(result_path):
        print(f'[{run_idx:02d}/{total}] {dataset} {subject_id}: '
              f'already done, skipping')
        continue

    print(f'\n[{run_idx:02d}/{total}] {dataset} {subject_id}')
    t_start = datetime.now()

    # ── Build training pool from all other subjects ────────
    X_train_list, y_train_list = [], []

    for ds2, sid2 in all_subjects:
        if ds2 == dataset and sid2 == subject_id:
            continue
        scores2, labels2, n_base2, _ = load_timeline(ds2, sid2)
        norm2 = normalise_timeline(scores2)
        X2, y2, _ = build_pairs(norm2, labels2, n_base2,
                                 FORECAST_WIN, FORECAST_AHEAD)
        if len(X2) > 0:
            X_train_list.append(X2)
            y_train_list.append(y2)

    X_train = np.concatenate(X_train_list, axis=0)
    y_train = np.concatenate(y_train_list, axis=0)

    n_pos = int(y_train.sum())
    n_neg = int((1 - y_train).sum())
    print(f'  Train pairs: {len(X_train)} '
          f'(pos={n_pos}, neg={n_neg}, '
          f'ratio=1:{n_neg//max(n_pos,1)})')

    # ── Build test pairs for this subject ──────────────────
    scores, labels, n_base, n_stress = load_timeline(dataset, subject_id)
    norm_scores = normalise_timeline(scores)
    X_test, y_test, positions = build_pairs(
        norm_scores, labels, n_base, FORECAST_WIN, FORECAST_AHEAD)

    print(f'  Test pairs : {len(X_test)} '
          f'(pos={int(y_test.sum())}, '
          f'neg={int((1-y_test).sum())})')

    # ── Train forecasting LSTM ─────────────────────────────
    model = ForecastingLSTM(input_size=1, hidden_size=32, n_layers=1)
    model, losses = train_forecaster(
        model, X_train, y_train,
        epochs=50, batch_size=32, lr=1e-3, patience=10
    )
    print(f'  Epochs run : {len(losses)}  '
          f'(loss {losses[0]:.4f} → {losses[-1]:.4f})')

    # ── Evaluate ───────────────────────────────────────────
    if len(X_test) == 0 or y_test.sum() == 0:
        print(f'  ⚠️  No test pairs or no positive labels')
        continue

    probs   = predict(model, X_test)
    y_pred  = (probs >= 0.5).astype(int)

    try:
        auroc = float(roc_auc_score(y_test, probs))
    except Exception:
        auroc = None

    f1    = float(f1_score(y_test, y_pred, zero_division=0))
    prec  = float(precision_score(y_test, y_pred, zero_division=0))
    rec   = float(recall_score(y_test, y_pred, zero_division=0))
    far   = float(np.mean(y_pred[y_test == 0])) if (y_test == 0).any() else 0.0

    # EWT
    ewt_s = compute_ewt(probs, positions, n_base,
                        FORECAST_WIN, threshold=0.5,
                        window_step_s=WINDOW_STEP_S)
    ewt_min = round(ewt_s / 60, 2) if ewt_s is not None else None

    elapsed = (datetime.now() - t_start).total_seconds()

    auroc_str = f'{auroc:.4f}' if auroc is not None else 'N/A'
    ewt_str   = f'{ewt_min} min' if ewt_min is not None else 'no alert'
    print(f'  AUROC={auroc_str}  F1={f1:.4f}  '
          f'FAR={far:.4f}  EWT={ewt_str}  ({elapsed:.1f}s)')

    result = {
        'dataset'       : dataset,
        'subject'       : subject_id,
        'run_index'     : run_idx,
        'timestamp'     : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'elapsed_sec'   : round(elapsed, 1),
        'n_train_pairs' : int(len(X_train)),
        'n_pos_train'   : int(n_pos),
        'n_neg_train'   : int(n_neg),
        'n_test_pairs'  : int(len(X_test)),
        'n_base'        : int(n_base),
        'n_stress'      : int(n_stress),
        'actual_epochs' : len(losses),
        'final_loss'    : float(losses[-1]),
        'metrics': {
            'AUROC'    : float(auroc) if auroc is not None else None,
            'F1'       : float(f1),
            'precision': float(prec),
            'recall'   : float(rec),
            'FAR'      : float(far),
        },
        'EWT_seconds'   : float(ewt_s) if ewt_s is not None else None,
        'EWT_minutes'   : float(ewt_min) if ewt_min is not None else None,
    }

    with open(result_path, 'w') as f:
        json.dump(result, f, indent=2)
    completed_runs = True

# ============================================================
# COMBINE + SUMMARISE
# ============================================================
print(f'\n{"="*60}')
print('Combining forecasting results...')

all_results = []
missing     = []

for dataset, subject_id in all_subjects:
    rpath = os.path.join(RESULTS_07,
                         f'{dataset}_{subject_id}_forecast.json')
    if os.path.exists(rpath):
        with open(rpath) as f:
            all_results.append(json.load(f))
    else:
        missing.append(f'{dataset}_{subject_id}')

if missing:
    print(f'⚠️  Missing: {missing}')
    print('Re-run this cell to complete.')
else:
    wesad_aurocs = [r['metrics']['AUROC'] for r in all_results
                    if r['dataset'] == 'WESAD'
                    and r['metrics']['AUROC'] is not None]
    aroad_aurocs = [r['metrics']['AUROC'] for r in all_results
                    if r['dataset'] == 'AROAD'
                    and r['metrics']['AUROC'] is not None]
    all_aurocs   = wesad_aurocs + aroad_aurocs

    wesad_f1s = [r['metrics']['F1'] for r in all_results
                 if r['dataset'] == 'WESAD']
    aroad_f1s = [r['metrics']['F1'] for r in all_results
                 if r['dataset'] == 'AROAD']

    ewts = [r['EWT_minutes'] for r in all_results
            if r['EWT_minutes'] is not None]

    summary = {
        'generated'            : datetime.now().strftime('%Y-%m-%d %H:%M'),
        'FORECAST_WIN'         : FORECAST_WIN,
        'FORECAST_AHEAD'       : FORECAST_AHEAD,
        'total_runs'           : len(all_results),
        'WESAD_AUROC_mean'     : round(float(np.mean(wesad_aurocs)), 4),
        'WESAD_AUROC_std'      : round(float(np.std(wesad_aurocs)), 4),
        'WESAD_F1_mean'        : round(float(np.mean(wesad_f1s)), 4),
        'AROAD_AUROC_mean'     : round(float(np.mean(aroad_aurocs)), 4),
        'AROAD_AUROC_std'      : round(float(np.std(aroad_aurocs)), 4),
        'AROAD_F1_mean'        : round(float(np.mean(aroad_f1s)), 4),
        'combined_AUROC_mean'  : round(float(np.mean(all_aurocs)), 4),
        'combined_AUROC_std'   : round(float(np.std(all_aurocs)), 4),
        'combined_F1_mean'     : round(float(np.mean(wesad_f1s + aroad_f1s)), 4),
        'EWT_mean_minutes'     : round(float(np.mean(ewts)), 2) if ewts else None,
        'EWT_std_minutes'      : round(float(np.std(ewts)), 2)  if ewts else None,
        'EWT_subjects_with_alert': len(ewts),
        'EWT_subjects_no_alert'  : len(all_results) - len(ewts),
        'individual_results'   : all_results,
    }

    summary_path = os.path.join(RESULTS_07, 'forecasting_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print(f'\n{"="*60}')
    print('FORECASTING COMPLETE — Summary')
    print('='*60)
    print(f'\n  WESAD  AUROC : {summary["WESAD_AUROC_mean"]:.4f} '
          f'± {summary["WESAD_AUROC_std"]:.4f}  '
          f'F1={summary["WESAD_F1_mean"]:.4f}')
    print(f'  AROAD  AUROC : {summary["AROAD_AUROC_mean"]:.4f} '
          f'± {summary["AROAD_AUROC_std"]:.4f}  '
          f'F1={summary["AROAD_F1_mean"]:.4f}')
    print(f'  Combined AUROC: {summary["combined_AUROC_mean"]:.4f} '
          f'± {summary["combined_AUROC_std"]:.4f}')
    print(f'\n  EWT mean     : {summary["EWT_mean_minutes"]} min '
          f'± {summary["EWT_std_minutes"]} min')
    print(f'  Subjects with alert before onset : '
          f'{summary["EWT_subjects_with_alert"]}/{len(all_results)}')
    print(f'  Subjects with no alert           : '
          f'{summary["EWT_subjects_no_alert"]}/{len(all_results)}')
    print(f'\n  Summary saved: forecasting_summary.json')
    print(f'\n{"="*60}')
    print('✅ Notebook 07 complete.')
    print('   All three notebooks done. Ready for dissertation write-up.')
    print('='*60)

Device: cpu

Total forecasting runs: 25
Input window: 10 scores (~5 min history)
Predict ahead: 10 windows (~5 min)

[01/25] WESAD S2: already done, skipping
[02/25] WESAD S4: already done, skipping
[03/25] WESAD S5: already done, skipping
[04/25] WESAD S7: already done, skipping
[05/25] WESAD S8: already done, skipping

[06/25] WESAD S9
  Train pairs: 1725 (pos=1065, neg=660, ratio=1:0)
  Test pairs : 48 (pos=29, neg=19)
  Epochs run : 50  (loss 0.6674 → 0.3948)
  AUROC=0.8675  F1=0.7458  FAR=0.4211  EWT=8.5 min  (7.8s)
[07/25] WESAD S10: already done, skipping
[08/25] WESAD S11: already done, skipping
[09/25] WESAD S13: already done, skipping
[10/25] WESAD S14: already done, skipping
[11/25] WESAD S15: already done, skipping
[12/25] WESAD S16: already done, skipping
[13/25] WESAD S17: already done, skipping
[14/25] AROAD Drv1: already done, skipping
[15/25] AROAD Drv2: already done, skipping
[16/25] AROAD Drv3: already done, skipping
[17/25] AROAD Drv5: already done, skipping
[18/25]

In [5]:
# ============================================================
# CELL 3b — Fix corrupt S9 file + rerun + re-combine
# ============================================================

import os
import json
import numpy as np

BASE       = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs'
RESULTS_07 = os.path.join(BASE, 'Forecasting', 'results')

# ── Step 1: Find and delete corrupt JSON files ─────────────
print('Checking all result files for corruption...')

corrupt = []
for fname in sorted(os.listdir(RESULTS_07)):
    if not fname.endswith('.json'):
        continue
    fpath = os.path.join(RESULTS_07, fname)
    try:
        with open(fpath) as f:
            json.load(f)
        print(f'  ✅ {fname}')
    except json.JSONDecodeError as e:
        print(f'  ❌ CORRUPT: {fname} — {e}')
        corrupt.append(fpath)

if corrupt:
    print(f'\nDeleting {len(corrupt)} corrupt file(s)...')
    for fpath in corrupt:
        os.remove(fpath)
        print(f'  Deleted: {os.path.basename(fpath)}')
    print('Re-run Cell 3 to regenerate missing results.')
else:
    print('\nNo corrupt files found.')
    print('The JSONDecodeError was from a file that is now fixed.')
    print('Run Cell 3 again to regenerate any missing files,')
    print('then the combining step will succeed.')

Checking all result files for corruption...
  ✅ AROAD_Drv10_forecast.json
  ✅ AROAD_Drv11_forecast.json
  ✅ AROAD_Drv12_forecast.json
  ✅ AROAD_Drv13_forecast.json
  ✅ AROAD_Drv1_forecast.json
  ✅ AROAD_Drv2_forecast.json
  ✅ AROAD_Drv3_forecast.json
  ✅ AROAD_Drv5_forecast.json
  ✅ AROAD_Drv6_forecast.json
  ✅ AROAD_Drv7_forecast.json
  ✅ AROAD_Drv8_forecast.json
  ✅ AROAD_Drv9_forecast.json
  ✅ WESAD_S10_forecast.json
  ✅ WESAD_S11_forecast.json
  ✅ WESAD_S13_forecast.json
  ✅ WESAD_S14_forecast.json
  ✅ WESAD_S15_forecast.json
  ✅ WESAD_S16_forecast.json
  ✅ WESAD_S17_forecast.json
  ✅ WESAD_S2_forecast.json
  ✅ WESAD_S4_forecast.json
  ✅ WESAD_S5_forecast.json
  ✅ WESAD_S7_forecast.json
  ✅ WESAD_S8_forecast.json
  ❌ CORRUPT: WESAD_S9_forecast.json — Expecting value: line 22 column 18 (char 506)

Deleting 1 corrupt file(s)...
  Deleted: WESAD_S9_forecast.json
Re-run Cell 3 to regenerate missing results.


In [7]:
# ============================================================
# CELL 3c — Fix combining step + print clean final summary
# ============================================================

import os
import json
import numpy as np
from datetime import datetime

BASE       = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs'
RESULTS_07 = os.path.join(BASE, 'Forecasting', 'results')

WESAD_STRESS_EXCLUDED = ['S3', 'S6']
WESAD_SUBJECTS        = ['S2','S3','S4','S5','S6','S7','S8','S9',
                         'S10','S11','S13','S14','S15','S16','S17']
AROAD_DRIVES          = ['Drv1','Drv2','Drv3','Drv5','Drv6','Drv7',
                         'Drv8','Drv9','Drv10','Drv11','Drv12','Drv13']
WESAD_EVAL = [s for s in WESAD_SUBJECTS if s not in WESAD_STRESS_EXCLUDED]
AROAD_EVAL = AROAD_DRIVES

all_subjects = (
    [('WESAD', sid) for sid in WESAD_EVAL] +
    [('AROAD', drv) for drv in AROAD_EVAL]
)

# Load all results
all_results = []
for dataset, subject_id in all_subjects:
    rpath = os.path.join(RESULTS_07,
                         f'{dataset}_{subject_id}_forecast.json')
    with open(rpath) as f:
        all_results.append(json.load(f))

# ── Extract metrics, using nanmean to skip Drv2 nan ───────
wesad_aurocs = [r['metrics']['AUROC'] for r in all_results
                if r['dataset'] == 'WESAD'
                and r['metrics']['AUROC'] is not None]
aroad_aurocs = [r['metrics']['AUROC'] for r in all_results
                if r['dataset'] == 'AROAD'
                and r['metrics']['AUROC'] is not None]

wesad_f1s = [r['metrics']['F1'] for r in all_results
             if r['dataset'] == 'WESAD']
aroad_f1s = [r['metrics']['F1'] for r in all_results
             if r['dataset'] == 'AROAD']

wesad_far = [r['metrics']['FAR'] for r in all_results
             if r['dataset'] == 'WESAD']
aroad_far = [r['metrics']['FAR'] for r in all_results
             if r['dataset'] == 'AROAD']

ewts = [r['EWT_minutes'] for r in all_results
        if r['EWT_minutes'] is not None]

# Use nanmean so Drv2 nan does not propagate
all_aurocs_valid = [a for a in wesad_aurocs + aroad_aurocs
                    if a is not None and not np.isnan(a)]

# ── Per-subject table ─────────────────────────────────────
print('='*70)
print('Per-subject forecasting results')
print('='*70)
print(f'\n{"Subject":<12} {"AUROC":>7} {"F1":>7} {"FAR":>7} '
      f'{"EWT":>10} {"Note"}')
print('-' * 65)

for r in all_results:
    sid    = f'{r["dataset"]} {r["subject"]}'
    auroc  = r['metrics']['AUROC']
    f1     = r['metrics']['F1']
    far    = r['metrics']['FAR']
    ewt    = r['EWT_minutes']

    auroc_s = f'{auroc:.4f}' if (auroc is not None and not np.isnan(auroc)) \
              else 'N/A'
    ewt_s   = f'{ewt:.1f} min' if ewt is not None else 'no alert'

    note = ''
    if r['dataset'] == 'AROAD' and r['subject'] == 'Drv2':
        note = '← no neg pairs'
    if far > 0.3:
        note = '← high FAR'

    print(f'{sid:<12} {auroc_s:>7} {f1:>7.4f} {far:>7.4f} '
          f'{ewt_s:>10}  {note}')

# ── Summary table ─────────────────────────────────────────
print(f'\n{"="*70}')
print('FORECASTING SUMMARY')
print('='*70)

print(f'\n  WESAD  AUROC : {np.nanmean(wesad_aurocs):.4f} '
      f'± {np.nanstd(wesad_aurocs):.4f}  '
      f'F1={np.mean(wesad_f1s):.4f}  '
      f'FAR={np.mean(wesad_far):.4f}')
print(f'  AROAD  AUROC : {np.nanmean(aroad_aurocs):.4f} '
      f'± {np.nanstd(aroad_aurocs):.4f}  '
      f'F1={np.mean(aroad_f1s):.4f}  '
      f'FAR={np.mean(aroad_far):.4f}')
print(f'  Combined AUROC: {np.nanmean(all_aurocs_valid):.4f} '
      f'± {np.nanstd(all_aurocs_valid):.4f}')
print(f'  Combined F1   : {np.mean(wesad_f1s + aroad_f1s):.4f}')

print(f'\n  Early Warning Time:')
print(f'    EWT mean   : {np.mean(ewts):.2f} min ± {np.std(ewts):.2f} min')
print(f'    EWT min    : {np.min(ewts):.1f} min')
print(f'    EWT max    : {np.max(ewts):.1f} min')
print(f'    Subjects with alert before onset : {len(ewts)}/25')
print(f'    Subjects with no alert fired     : {25 - len(ewts)}/25')

print(f'\n  Drv2 note: excluded from AUROC mean (zero negative test pairs')
print(f'  due to short baseline — all timeline positions labelled positive')
print(f'  within the forecast horizon).')

# ── Save corrected summary ────────────────────────────────
summary = {
    'generated'              : datetime.now().strftime('%Y-%m-%d %H:%M'),
    'FORECAST_WIN'           : 10,
    'FORECAST_AHEAD'         : 10,
    'total_runs'             : len(all_results),
    'WESAD_n_eval'           : len(wesad_aurocs),
    'AROAD_n_eval'           : len([a for a in aroad_aurocs
                                    if not np.isnan(a)]),
    'AROAD_Drv2_note'        : 'AUROC not computed — zero negative test pairs',
    'WESAD_AUROC_mean'       : round(float(np.nanmean(wesad_aurocs)), 4),
    'WESAD_AUROC_std'        : round(float(np.nanstd(wesad_aurocs)), 4),
    'WESAD_F1_mean'          : round(float(np.mean(wesad_f1s)), 4),
    'WESAD_FAR_mean'         : round(float(np.mean(wesad_far)), 4),
    'AROAD_AUROC_mean'       : round(float(np.nanmean(aroad_aurocs)), 4),
    'AROAD_AUROC_std'        : round(float(np.nanstd(aroad_aurocs)), 4),
    'AROAD_F1_mean'          : round(float(np.mean(aroad_f1s)), 4),
    'AROAD_FAR_mean'         : round(float(np.mean(aroad_far)), 4),
    'combined_AUROC_mean'    : round(float(np.nanmean(all_aurocs_valid)), 4),
    'combined_AUROC_std'     : round(float(np.nanstd(all_aurocs_valid)), 4),
    'combined_F1_mean'       : round(float(np.mean(wesad_f1s + aroad_f1s)), 4),
    'EWT_mean_minutes'       : round(float(np.mean(ewts)), 2),
    'EWT_std_minutes'        : round(float(np.std(ewts)), 2),
    'EWT_min_minutes'        : round(float(np.min(ewts)), 1),
    'EWT_max_minutes'        : round(float(np.max(ewts)), 1),
    'EWT_subjects_with_alert': len(ewts),
    'EWT_subjects_no_alert'  : 25 - len(ewts),
    'individual_results'     : all_results,
}

summary_path = os.path.join(RESULTS_07, 'forecasting_summary_final.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\n  Summary saved: forecasting_summary_final.json')
print(f'\n{"="*70}')
print('✅ Notebook 07 complete. All three notebooks done.')
print('='*70)

Per-subject forecasting results

Subject        AUROC      F1     FAR        EWT Note
-----------------------------------------------------------------
WESAD S2      0.8948  0.7826  0.0000   no alert  
WESAD S4      0.9540  0.7917  0.0000   no alert  
WESAD S5      0.7967  0.7917  0.0000   no alert  
WESAD S7      0.7913  0.7660  0.0000   no alert  
WESAD S8      0.7019  0.7500  0.0000   no alert  
WESAD S9      0.8675  0.7458  0.4211    8.5 min  ← high FAR
WESAD S10     0.8421  0.8148  0.0000   no alert  
WESAD S11     0.8930  0.8000  0.0000   no alert  
WESAD S13     0.8737  0.8000  0.0000   no alert  
WESAD S14     0.9825  0.8000  0.0000   no alert  
WESAD S15     0.8000  0.8000  0.0000   no alert  
WESAD S16     0.7421  0.8000  0.0000   no alert  
WESAD S17     0.9622  0.8148  0.0000   no alert  
AROAD Drv1    0.7188  0.7863  0.3077   13.5 min  ← high FAR
AROAD Drv2       N/A  0.7952  0.0000   no alert  ← no neg pairs
AROAD Drv3    0.8601  0.9000  0.0000   no alert  
AROAD Drv5    

In [8]:
# Quick threshold sensitivity check — run in notebook 07
import os, json, numpy as np

RESULTS_07 = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs/Forecasting/results'

with open(os.path.join(RESULTS_07, 'forecasting_summary_final.json')) as f:
    summary = json.load(f)

results = summary['individual_results']

for threshold in [0.5, 0.4, 0.3]:
    ewts, alerts = [], 0
    for r in results:
        # Recompute EWT at this threshold is not possible without
        # re-running inference. Instead check if existing EWT
        # subjects change count.
        pass

print('To properly test lower thresholds, re-run Cell 3 with threshold=0.3')
print('Change: y_pred = (probs >= 0.3).astype(int)')
print('And in compute_ewt: threshold=0.3')
print('Everything else stays the same.')
print('Re-run Cell 3 with this change, all 25 results will regenerate.')

To properly test lower thresholds, re-run Cell 3 with threshold=0.3
Change: y_pred = (probs >= 0.3).astype(int)
And in compute_ewt: threshold=0.3
Everything else stays the same.
Re-run Cell 3 with this change, all 25 results will regenerate.
